<a href="https://colab.research.google.com/github/L-Poca/Data_Pipeline/blob/rafael_cleaning/notebooks/colab/inceptionv3_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🦠 InceptionV3 Training - COVID-19 Classification

---

## 📋 Overview

Ce notebook entraîne **uniquement le modèle InceptionV3** pour la classification COVID-19.

### 🎯 Pipeline

1. **Chargement des données** avec preprocessing RGB
2. **Entraînement Phase 1**: Feature extraction (base gelée)
3. **Entraînement Phase 2**: Fine-tuning (30 top layers dégelées)
4. **Évaluation complète**: Métriques, matrices, courbes
5. **Interprétabilité**: Grad-CAM + LIME
6. **Sauvegarde**: Modèle final + résultats

### 📊 Dataset

- **Classes**: COVID, Lung_Opacity, Normal, Viral Pneumonia
- **Images**: ~21,000 chest X-rays (grayscale → RGB)
- **Challenge**: Class imbalance

### ⏱️ Estimated Runtime

- **Fast mode** (100 images/class): ~15-30 minutes
- **Full mode** (all images): ~1-2 hours (with GPU)

---

In [ ]:
"""
╔════════════════════════════════════════════════════════════════════════════╗
║  🎯 CELLULE DE CONFIGURATION STANDALONE - COPIER-COLLER DANS VOS NOTEBOOKS ║
╚════════════════════════════════════════════════════════════════════════════╝

INSTRUCTIONS:
-------------
1. Copiez TOUT le contenu de cette cellule
2. Collez-le comme PREMIÈRE CELLULE de votre notebook
3. Exécutez la cellule
4. La configuration est prête à l'emploi !

Cette cellule est 100% autonome et fonctionne partout :
✅ Google Colab (clone + installe automatiquement)
✅ WSL / Linux Local
✅ Tout environnement Jupyter

APRÈS EXÉCUTION, UTILISEZ L'OBJET 'config':
--------------------------------------------
▶ config.data_dir              # Chemin du dataset
▶ config.models_dir            # Répertoire des modèles
▶ config.results_dir           # Répertoire des résultats
▶ config.classes               # Liste des classes
▶ config.img_size              # Tuple (width, height)
▶ config.img_channels          # Nombre de canaux (1=grayscale, 3=RGB)
▶ config.batch_size            # Taille des batchs
▶ config.epochs                # Nombre d'époques
▶ config.learning_rate         # Learning rate
▶ config.validation_split      # Proportion pour validation
▶ config.gradcam_alpha         # Alpha pour Grad-CAM
▶ config.shap_max_evals        # Evaluations SHAP
▶ config.confidence_high_threshold  # Seuil confiance haute
... et bien plus !

VARIABLES GLOBALES:
-------------------
• config: Objet Config complet (tous les paramètres du projet)
• ENV: Environnement détecté ('colab', 'wsl', 'local')
• Tous les transformers importés et prêts à l'emploi

"""

# =============================================================================
# IMPORTS STANDARDS
# =============================================================================

import os
import sys
import subprocess
from pathlib import Path


# =============================================================================
# DÉTECTION AUTOMATIQUE DE L'ENVIRONNEMENT
# =============================================================================

def detect_environment():
    """Détecte l'environnement (colab, wsl, local)"""
    try:
        import google.colab
        return "colab"
    except ImportError:
        is_wsl = os.path.exists('/proc/version') and 'microsoft' in open('/proc/version').read().lower()
        return "wsl" if is_wsl else "local"

ENV = detect_environment()
print(f"🌍 Environnement: {ENV.upper()}")


# =============================================================================
# BOOTSTRAP COLAB (Clone + Install si nécessaire)
# =============================================================================

if ENV == "colab":
    print("\n🚀 Bootstrap Colab...")
    
    os.chdir('/content')
    if not os.path.exists('/content/Data_Pipeline'):
        print("📥 Clonage du repository...")
        subprocess.run(['git', 'clone', 'https://github.com/L-Poca/Data_Pipeline.git'], check=True)
    
    os.chdir('/content/Data_Pipeline')
    
    # Checkout de la branche rafael_cleaning
    result = subprocess.run(
        ['git', 'checkout', '-b', 'rafael_cleaning', 'origin/rafael_cleaning'],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        # Si la branche locale existe déjà, juste switcher
        subprocess.run(['git', 'checkout', 'rafael_cleaning'], capture_output=True)
    
    # Installation du package en mode éditable (sans dépendances - détection Colab dans setup.py)
    print("📦 Installation du package...")
    result = subprocess.run(['pip', 'install', '-e', '.', '--quiet'], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"⚠️ Erreur installation: {result.stderr}")
    else:
        print("✅ Package installé")
    
    print("💾 Montage Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Extraction dataset
    archive_data = '/content/drive/MyDrive/DS_COVID/archive_covid.zip'
    if os.path.exists(archive_data):
        print("📦 Extraction dataset...")
        os.makedirs('./data/raw/', exist_ok=True)
        subprocess.run(['unzip', '-o', '-q', archive_data, '-d', './data/raw/COVID-19_Radiography_Dataset/'])
    
    # Extraction models
    archive_models = '/content/drive/MyDrive/DS_COVID/inceptionv3_best.zip'
    if os.path.exists(archive_models):
        print("📦 Extraction models...")
        os.makedirs('./models/', exist_ok=True)
        subprocess.run(['unzip', '-o', '-q', archive_models, '-d', './models/'])

    print("✅ Bootstrap terminé")


# =============================================================================
# CONFIGURATION DES CHEMINS
# =============================================================================

# Déterminer project_root selon l'environnement
if ENV == "colab":
    project_root = Path('/content/Data_Pipeline')
elif ENV == "wsl":
    project_root = Path('/home/cepa/DST/projet_DS/Data_Pipeline/Data_Pipeline')
else:  # local
    # Depuis un notebook dans src/notebooks/
    project_root = Path.cwd().parent.parent

# Vérification du modèle en local (WSL ou autre)
if ENV != "colab":
    models_dir = project_root / 'models'
    model_path = models_dir / 'inceptionv3_best.keras'
    
    if model_path.exists():
        print(f"✅ Modèle InceptionV3 trouvé: {model_path}")
    else:
        print(f"⚠️ Modèle InceptionV3 non trouvé: {model_path}")
        print(f"   Veuillez placer inceptionv3_best.keras dans {models_dir}/")

# Ajouter src/ au sys.path pour les imports
# src_path = str(project_root / 'src')
# if src_path not in sys.path:
#     sys.path.insert(0, src_path)
#     print(f"✅ Chemin src/ ajouté: {src_path}")

# Charger la configuration depuis JSON
from src.utils.config import build_config

config = build_config(project_root, ENV)

print(f"\n🎯 Configuration chargée depuis config/{ENV}_config.json")


# =============================================================================
# IMPORTS DES TRANSFORMERS
# =============================================================================

try:
    from src.features.Pipelines.Transformateurs.image_loaders import ImageLoader
    from src.features.Pipelines.Transformateurs.image_preprocessing import (
        ImageResizer, ImageNormalizer, ImageFlattener, ImageMasker
    )
    from src.features.Pipelines.Transformateurs.image_augmentation import (
        ImageAugmenter, ImageRandomCropper
    )
    from src.features.Pipelines.Transformateurs.image_features import (
        ImageHistogram, ImagePCA, ImageStandardScaler
    )
    print("✅ Transformers importés")
except ImportError as e:
    print(f"⚠️ Erreur import transformers: {e}")


# =============================================================================
# IMPORTS ML/DL
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras

# =============================================================================
# CONFIGURATION MATPLOTLIB (utilise config pour les paramètres)
# =============================================================================

plt.rcParams['figure.figsize'] = config.figure_size
plt.rcParams['figure.dpi'] = config.dpi
plt.style.use(config.plot_style)
sns.set_palette(config.color_palette)

# =============================================================================
# AFFICHAGE DU RÉSUMÉ
# =============================================================================

print("\n" + "=" * 80)
print("✅ CONFIGURATION PRÊTE - Data Pipeline")
print("=" * 80)
print(f"📂 Projet:       {config.project_root}")
print(f"📊 Dataset:      {config.data_dir}")
print(f"💾 Modèles:      {config.models_dir}")
print(f"📈 Résultats:    {config.results_dir}")
print(f"📐 Dataset:      {'✅ Accessible' if config.data_dir.exists() else '❌ Introuvable'}")
print()
print(f"🏷️  Classes:     {', '.join(config.classes)} ({config.num_classes} classes)")
print(f"🎛️  Images:      {config.img_size} | {config.img_channels} canaux")
print(f"🔧 Training:     Batch={config.batch_size} | Epochs={config.epochs} | LR={config.learning_rate}")
print(f"� Splits:       Train/Val={1-config.validation_split:.0%} | Val={config.validation_split:.0%} | Test={config.test_split:.0%}")
print()
print(f"🎨 Viz:          Style={config.plot_style} | Palette={config.color_palette}")
print(f"📏 Figures:      {config.figure_size} @ {config.dpi} DPI")
print()
print(f"🔍 Interprét.:   GradCAM α={config.gradcam_alpha} | SHAP evals={config.shap_max_evals}")
print(f"📉 Seuils conf.: High={config.confidence_high_threshold} | Medium={config.confidence_medium_threshold}")
print("=" * 80)
print("\n💡 Variable principale:")
print("   • config: Objet Config complet (accès à TOUS les paramètres)")
print("   • ENV: Environnement actuel")
print()
print("📚 Exemples d'utilisation:")
print("   config.data_dir          # Chemin du dataset")
print("   config.classes           # Liste des classes")
print("   config.img_size          # Tuple (width, height)")
print("   config.batch_size        # Taille des batchs")
print("   config.models_dir        # Répertoire des modèles")
print("   config.gradcam_alpha     # Paramètres d'interprétabilité")
print()
print("🎯 Transformers disponibles:")
print("   • ImageLoader, ImageResizer, ImageNormalizer, ImageFlattener, ImageMasker")
print("   • ImageAugmenter, ImageRandomCropper")
print("   • ImageHistogram, ImagePCA, ImageStandardScaler")
print("=" * 80)


In [ ]:
if ENV == 'colab' :
  !pip install optuna

## 🔧 Section 1: Configuration du Dataset

Choisissez le nombre d'images par classe pour l'entraînement.

In [ ]:
# =============================================================================
# CONFIGURATION DU DATASET
# =============================================================================

# Pour test rapide: 100 images par classe
# Pour entraînement complet: None (toutes les images)
N_IMAGES_PER_CLASS = None  # Mettre 100 pour test rapide

print("=" * 70)
print("CONFIGURATION DU DATASET")
print("=" * 70)

if N_IMAGES_PER_CLASS:
    print(f"\n⚡ Mode RAPIDE: {N_IMAGES_PER_CLASS} images par classe")
    print(f"   Total attendu: ~{N_IMAGES_PER_CLASS * len(config.classes)} images")
else:
    print(f"\n🚀 Mode COMPLET: Toutes les images")
    print(f"   Total attendu: ~21,000 images")

print(f"\n✅ Configuration prête")

## 📊 Section 2: Chargement des Données

Chargement du dataset et conversion en RGB pour InceptionV3.

In [ ]:
# =============================================================================
# CHARGEMENT DES DONNÉES
# =============================================================================

from src.notebooks import load_dataset, create_preprocessing_pipeline

print("\n" + "=" * 70)
print("CHARGEMENT DES DONNÉES")
print("=" * 70)

# Charger les chemins des images
image_paths, mask_paths, labels, labels_int = load_dataset(
    data_dir=config.data_dir,
    categories=config.classes,
    n_images_per_class=N_IMAGES_PER_CLASS,
    load_masks=False,  # Pas besoin des masques pour classification
    verbose=True
)

print(f"\n✅ Dataset chargé:")
print(f"   • Total images: {len(images)}")
print(f"   • Shape: {images.shape}")
print(f"   • Classes: {class_names}")
print(f"   • Distribution: {np.bincount(labels)}")

# Préparer les splits
X_train, X_val, X_test, y_train, y_val, y_test = prepare_train_val_test_split(
    images, labels,
    test_size=0.15,
    val_size=0.15,
    random_state=42
)

print(f"\n✅ Splits créés:")
print(f"   • Train: {len(X_train)} images")
print(f"   • Val: {len(X_val)} images")
print(f"   • Test: {len(X_test)} images")

# Convertir en RGB (InceptionV3 nécessite 3 canaux)
print(f"\n🔄 Conversion en RGB...")
X_train_rgb = np.repeat(X_train[..., np.newaxis], 3, axis=-1)
X_val_rgb = np.repeat(X_val[..., np.newaxis], 3, axis=-1)
X_test_rgb = np.repeat(X_test[..., np.newaxis], 3, axis=-1)

print(f"   • Train shape: {X_train_rgb.shape}")
print(f"   • Val shape: {X_val_rgb.shape}")
print(f"   • Test shape: {X_test_rgb.shape}")

# Convertir les labels en categorical
y_train_cat = to_categorical(y_train, num_classes=config.num_classes)
y_val_cat = to_categorical(y_val, num_classes=config.num_classes)
y_test_cat = to_categorical(y_test, num_classes=config.num_classes)

print(f"\n   • Labels shape: {y_train_cat.shape}")

# Calculer les poids de classes
class_weights_dict = compute_class_weights(y_train)

print(f"\n⚖️  Class weights:")
for cls_idx, weight in class_weights_dict.items():
    print(f"   • {config.classes[cls_idx]}: {weight:.2f}")

print("\n✅ Données prêtes pour l'entraînement!")

## 🏗️ Section 3: Construction du Modèle InceptionV3

Architecture Transfer Learning avec base pré-entraînée sur ImageNet.

In [ ]:
# =============================================================================
# CONSTRUCTION DU MODÈLE INCEPTIONV3
# =============================================================================

print("\n" + "=" * 70)
print("CONSTRUCTION DU MODÈLE INCEPTIONV3")
print("=" * 70)

# Construire le modèle
print("\n🔨 Architecture:")
model, base_model = build_transfer_learning_model(
    base_model_name='InceptionV3',
    input_shape=(config.img_size[0], config.img_size[1], 3),
    num_classes=config.num_classes,
    freeze_base=True,
    verbose=True
)

print("\n📊 Résumé du modèle:")
print(f"   • Total parameters: {model.count_params():,}")
print(f"   • Base model layers: {len(base_model.layers)}")
print(f"   • Base frozen: {not base_model.trainable}")

print("\n✅ Modèle construit!")

## 🚀 Section 4: Phase 1 - Feature Extraction

Entraînement avec la base gelée (20 epochs).

In [ ]:
# =============================================================================
# PHASE 1: FEATURE EXTRACTION (Base gelée)
# =============================================================================

print("\n" + "=" * 70)
print("PHASE 1: FEATURE EXTRACTION")
print("=" * 70)

# Compiler
print("\n⚙️  Compilation...")
model = compile_model(model, learning_rate=config.learning_rate, verbose=True)

# Callbacks
model_save_dir_p1 = config.models_dir / "InceptionV3_phase1"
callbacks_phase1 = create_callbacks(
    models_dir=model_save_dir_p1,
    patience_early_stop=10,
    patience_reduce_lr=5,
    monitor='val_accuracy',
    verbose=True
)

# Entraînement
print("\n🚀 Entraînement Phase 1...")
print(f"   • Epochs: 20")
print(f"   • Batch size: {config.batch_size}")
print(f"   • Learning rate: {config.learning_rate}")

start_time_p1 = time.time()

history_phase1 = model.fit(
    X_train_rgb, y_train_cat,
    validation_data=(X_val_rgb, y_val_cat),
    epochs=20,
    batch_size=config.batch_size,
    callbacks=callbacks_phase1,
    class_weight=class_weights_dict,
    verbose=1
)

train_time_p1 = time.time() - start_time_p1

print(f"\n✅ Phase 1 terminée en {train_time_p1:.2f} secondes")
print(f"   • Best val_accuracy: {max(history_phase1.history['val_accuracy']):.4f}")

## 🔓 Section 5: Phase 2 - Fine-Tuning

Dégel des 30 top layers et fine-tuning.

In [ ]:
# =============================================================================
# PHASE 2: FINE-TUNING (Top layers dégelées)
# =============================================================================

print("\n" + "=" * 70)
print("PHASE 2: FINE-TUNING")
print("=" * 70)

# Dégeler les top layers
print("\n🔓 Dégel des top layers...")
model = unfreeze_top_layers(
    base_model=base_model,
    model=model,
    n_layers=30,
    learning_rate=config.learning_rate / 10,
    verbose=True
)

# Callbacks
model_save_dir_p2 = config.models_dir / "InceptionV3_phase2"
callbacks_phase2 = create_callbacks(
    models_dir=model_save_dir_p2,
    patience_early_stop=15,
    patience_reduce_lr=7,
    monitor='val_accuracy',
    verbose=True
)

# Entraînement
print("\n🚀 Entraînement Phase 2...")
print(f"   • Epochs: {config.epochs}")
print(f"   • Batch size: {config.batch_size}")
print(f"   • Learning rate: {config.learning_rate / 10}")

start_time_p2 = time.time()

history_phase2 = model.fit(
    X_train_rgb, y_train_cat,
    validation_data=(X_val_rgb, y_val_cat),
    epochs=config.epochs,
    batch_size=config.batch_size,
    callbacks=callbacks_phase2,
    class_weight=class_weights_dict,
    verbose=1
)

train_time_p2 = time.time() - start_time_p2

print(f"\n✅ Phase 2 terminée en {train_time_p2:.2f} secondes")
print(f"   • Best val_accuracy: {max(history_phase2.history['val_accuracy']):.4f}")

total_train_time = train_time_p1 + train_time_p2
print(f"\n⏱️  Temps d'entraînement total: {total_train_time:.2f}s ({total_train_time/60:.2f} min)")

## 📊 Section 6: Évaluation sur le Test Set

Métriques complètes : Accuracy, F1, Precision, Recall.

In [ ]:
# =============================================================================
# ÉVALUATION SUR LE TEST SET
# =============================================================================

print("\n" + "=" * 70)
print("ÉVALUATION SUR LE TEST SET")
print("=" * 70)

# Prédictions
print("\n🔮 Prédictions...")
y_pred_probs = model.predict(X_test_rgb, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# Métriques de base
eval_results = model.evaluate(X_test_rgb, y_test_cat, verbose=0)
test_loss = eval_results[0]
test_acc = eval_results[1]

# F1 Score
f1_weighted = f1_score(y_test, y_pred, average='weighted')
f1_per_class = f1_score(y_test, y_pred, average=None)

# Precision, Recall
precision, recall, _, _ = precision_recall_fscore_support(
    y_test, y_pred, average='weighted'
)

print(f"\n📊 RÉSULTATS FINAUX:")
print(f"   • Test Loss: {test_loss:.4f}")
print(f"   • Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"   • F1 Score (weighted): {f1_weighted:.4f}")
print(f"   • Precision (weighted): {precision:.4f}")
print(f"   • Recall (weighted): {recall:.4f}")

print(f"\n📋 F1 Score par classe:")
for i, class_name in enumerate(config.classes):
    print(f"   • {class_name}: {f1_per_class[i]:.4f}")

# Rapport de classification
print(f"\n📄 Classification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=config.classes,
    digits=4
))

## 📈 Section 7: Visualisation des Courbes d'Entraînement

Évolution de la loss et de l'accuracy pour les 2 phases.

In [ ]:
# =============================================================================
# COURBES D'ENTRAÎNEMENT
# =============================================================================

print("\n" + "=" * 70)
print("VISUALISATION DES COURBES D'ENTRAÎNEMENT")
print("=" * 70)

results_dir = config.results_dir / 'inceptionv3'
results_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Phase 1 - Loss
axes[0, 0].plot(history_phase1.history['loss'], label='Train Loss', linewidth=2)
axes[0, 0].plot(history_phase1.history['val_loss'], label='Val Loss', linewidth=2)
axes[0, 0].set_title('Phase 1: Loss', fontsize=12, weight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Phase 1 - Accuracy
axes[0, 1].plot(history_phase1.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0, 1].plot(history_phase1.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0, 1].set_title('Phase 1: Accuracy', fontsize=12, weight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Phase 2 - Loss
axes[1, 0].plot(history_phase2.history['loss'], label='Train Loss', linewidth=2)
axes[1, 0].plot(history_phase2.history['val_loss'], label='Val Loss', linewidth=2)
axes[1, 0].set_title('Phase 2: Loss', fontsize=12, weight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Phase 2 - Accuracy
axes[1, 1].plot(history_phase2.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[1, 1].plot(history_phase2.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[1, 1].set_title('Phase 2: Accuracy', fontsize=12, weight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.suptitle('InceptionV3 - Training Curves', fontsize=14, weight='bold')
plt.tight_layout()
plt.savefig(results_dir / 'training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Courbes sauvegardées: {results_dir / 'training_curves.png'}")

## 🎯 Section 8: Matrice de Confusion

Visualisation des prédictions correctes et incorrectes.

In [ ]:
# =============================================================================
# MATRICE DE CONFUSION
# =============================================================================

print("\n" + "=" * 70)
print("MATRICE DE CONFUSION")
print("=" * 70)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=config.classes,
    yticklabels=config.classes,
    cbar_kws={'label': 'Number of Predictions'}
)
plt.title('InceptionV3 - Confusion Matrix', fontsize=14, weight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig(results_dir / 'confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Matrice sauvegardée: {results_dir / 'confusion_matrix.png'}")

# Statistiques de la matrice
print(f"\n📊 Statistiques par classe:")
for i, class_name in enumerate(config.classes):
    correct = cm[i, i]
    total = cm[i, :].sum()
    accuracy_class = correct / total if total > 0 else 0
    print(f"   • {class_name}: {correct}/{total} ({accuracy_class*100:.2f}%)")

## 📉 Section 9: Courbes ROC

ROC curves et AUC pour chaque classe (One-vs-Rest).

In [ ]:
# =============================================================================
# COURBES ROC
# =============================================================================

print("\n" + "=" * 70)
print("COURBES ROC")
print("=" * 70)

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

for i, class_name in enumerate(config.classes):
    # Binariser les labels
    y_test_binary = (y_test == i).astype(int)
    y_pred_binary = y_pred_probs[:, i]
    
    # Calculer ROC
    fpr, tpr, _ = roc_curve(y_test_binary, y_pred_binary)
    roc_auc = auc(fpr, tpr)
    
    # Plot
    axes[i].plot(fpr, tpr, label=f'AUC = {roc_auc:.3f}', linewidth=2, color='darkorange')
    axes[i].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
    axes[i].set_title(f'{class_name}', fontsize=12, weight='bold')
    axes[i].set_xlabel('False Positive Rate', fontsize=10)
    axes[i].set_ylabel('True Positive Rate', fontsize=10)
    axes[i].legend(loc='lower right')
    axes[i].grid(alpha=0.3)
    
    print(f"   • {class_name}: AUC = {roc_auc:.4f}")

plt.suptitle('InceptionV3 - ROC Curves', fontsize=14, weight='bold')
plt.tight_layout()
plt.savefig(results_dir / 'roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✅ Courbes ROC sauvegardées: {results_dir / 'roc_curves.png'}")

## 🔍 Section 10: Analyse d'Interprétabilité

Grad-CAM et LIME pour comprendre les décisions du modèle.

In [ ]:
# =============================================================================
# INTERPRÉTABILITÉ - GRAD-CAM + LIME
# =============================================================================

print("\n" + "=" * 70)
print("ANALYSE D'INTERPRÉTABILITÉ")
print("=" * 70)

interpretability_dir = results_dir / 'interpretability'
interpretability_dir.mkdir(parents=True, exist_ok=True)

# Preprocessing function pour InceptionV3
try:
    preprocess_fn = get_preprocessing_function('InceptionV3')
    print("   ✅ Preprocessing function: InceptionV3")
except Exception as e:
    print(f"   ⚠️  Preprocessing function non trouvée: {e}")
    preprocess_fn = None

# Analyse complète
print("\n🔍 Lancement de l'analyse (Grad-CAM + LIME)...")
print(f"   • Échantillons: 2 par classe")
print(f"   • Stratégie: one_per_class")

try:
    run_full_interpretability_analysis(
        model=model,
        x_data=X_test_rgb,
        y_true=y_test,
        y_pred=y_pred,
        class_names=config.classes,
        background_data=X_train_rgb[:50],
        n_samples=2,
        strategy="one_per_class",
        save_dir=interpretability_dir,
        use_gradcam=True,
        use_lime=True,
        use_shap=False,  # SHAP désactivé (trop lent)
        preprocess_fn=preprocess_fn
    )
    
    print(f"\n✅ Interprétabilité sauvegardée dans: {interpretability_dir}")
    print(f"   • Grad-CAM: {interpretability_dir / 'gradcam'}")
    print(f"   • LIME: {interpretability_dir / 'lime'}")
    
except Exception as e:
    print(f"\n⚠️  Erreur lors de l'analyse d'interprétabilité: {e}")
    print("   Continuez sans interprétabilité ou installez:")
    print("   pip install lime shap")

## ❌ Section 11: Analyse des Erreurs

Identification et visualisation des erreurs de classification.

In [ ]:
# =============================================================================
# ANALYSE DES ERREURS
# =============================================================================

print("\n" + "=" * 70)
print("ANALYSE DES ERREURS")
print("=" * 70)

# Trouver les erreurs
errors = y_test != y_pred
error_indices = np.where(errors)[0]

print(f"\n📊 Statistiques d'erreurs:")
print(f"   • Total errors: {errors.sum()}/{len(y_test)} ({errors.sum()/len(y_test)*100:.2f}%)")
print(f"   • Correct: {(~errors).sum()}/{len(y_test)} ({(~errors).sum()/len(y_test)*100:.2f}%)")

# Matrice d'erreurs
error_matrix = np.zeros((config.num_classes, config.num_classes), dtype=int)
for true_label, pred_label in zip(y_test[errors], y_pred[errors]):
    error_matrix[true_label, pred_label] += 1

print("\n📋 Error Matrix (True → Predicted):")
print(f"{'':>20s}", end="")
for cls in config.classes:
    print(f"{cls:>20s}", end="")
print()

for i, cls in enumerate(config.classes):
    print(f"{cls:>20s}", end="")
    for j in range(len(config.classes)):
        print(f"{error_matrix[i, j]:>20d}", end="")
    print()

# Visualiser les erreurs
if len(error_indices) > 0:
    print(f"\n📸 Visualisation des erreurs...")
    
    n_errors_to_show = min(20, len(error_indices))
    sample_error_indices = error_indices[:n_errors_to_show]
    
    fig, axes = plt.subplots(4, 5, figsize=(15, 12))
    axes = axes.ravel()
    
    for idx, err_idx in enumerate(sample_error_indices):
        axes[idx].imshow(X_test_rgb[err_idx, :, :, 0], cmap='gray')
        true_cls = config.classes[y_test[err_idx]]
        pred_cls = config.classes[y_pred[err_idx]]
        conf = y_pred_probs[err_idx, y_pred[err_idx]]
        axes[idx].set_title(f'T: {true_cls}\nP: {pred_cls} ({conf:.2f})', fontsize=8)
        axes[idx].axis('off')
    
    # Cacher les axes inutilisés
    for idx in range(n_errors_to_show, 20):
        axes[idx].axis('off')
    
    plt.suptitle('Top 20 Misclassified Samples', fontsize=14, weight='bold')
    plt.tight_layout()
    plt.savefig(results_dir / 'error_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✅ Analyse d'erreurs sauvegardée: {results_dir / 'error_analysis.png'}")
else:
    print("\n🎉 Aucune erreur trouvée!")

## 💾 Section 12: Sauvegarde du Modèle et des Résultats

Sauvegarde du modèle final et d'un rapport de résultats.

In [ ]:
# =============================================================================
# SAUVEGARDE DU MODÈLE ET DES RÉSULTATS
# =============================================================================

print("\n" + "=" * 70)
print("SAUVEGARDE")
print("=" * 70)

# Sauvegarder le modèle
model_path = config.models_dir / 'inceptionv3_final.keras'
model.save(model_path)
print(f"\n✅ Modèle sauvegardé: {model_path}")

# Sauvegarder les résultats dans un fichier texte
results_file = results_dir / 'results.txt'

with open(results_file, 'w') as f:
    f.write("=" * 70 + "\n")
    f.write("INCEPTIONV3 - RÉSULTATS FINAUX\n")
    f.write("=" * 70 + "\n\n")
    
    f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    
    f.write("Configuration:\n")
    f.write(f"  • Image size: {config.img_size}\n")
    f.write(f"  • Batch size: {config.batch_size}\n")
    f.write(f"  • Learning rate: {config.learning_rate}\n")
    f.write(f"  • Epochs Phase 1: 20\n")
    f.write(f"  • Epochs Phase 2: {config.epochs}\n")
    f.write(f"  • Images per class: {N_IMAGES_PER_CLASS if N_IMAGES_PER_CLASS else 'All'}\n\n")
    
    f.write("Dataset:\n")
    f.write(f"  • Train: {len(X_train_rgb)} images\n")
    f.write(f"  • Val: {len(X_val_rgb)} images\n")
    f.write(f"  • Test: {len(X_test_rgb)} images\n\n")
    
    f.write("Temps d'entraînement:\n")
    f.write(f"  • Phase 1: {train_time_p1:.2f}s ({train_time_p1/60:.2f} min)\n")
    f.write(f"  • Phase 2: {train_time_p2:.2f}s ({train_time_p2/60:.2f} min)\n")
    f.write(f"  • Total: {total_train_time:.2f}s ({total_train_time/60:.2f} min)\n\n")
    
    f.write("Métriques:\n")
    f.write(f"  • Test Loss: {test_loss:.4f}\n")
    f.write(f"  • Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)\n")
    f.write(f"  • F1 Score (weighted): {f1_weighted:.4f}\n")
    f.write(f"  • Precision (weighted): {precision:.4f}\n")
    f.write(f"  • Recall (weighted): {recall:.4f}\n\n")
    
    f.write("F1 Score par classe:\n")
    for i, class_name in enumerate(config.classes):
        f.write(f"  • {class_name}: {f1_per_class[i]:.4f}\n")
    
    f.write("\n" + "=" * 70 + "\n")
    f.write("CLASSIFICATION REPORT\n")
    f.write("=" * 70 + "\n\n")
    f.write(classification_report(
        y_test, y_pred,
        target_names=config.classes,
        digits=4
    ))

print(f"✅ Résultats sauvegardés: {results_file}")

# Résumé final
print("\n" + "=" * 70)
print("✅ ENTRAÎNEMENT TERMINÉ")
print("=" * 70)

print(f"\n📊 RÉSUMÉ:")
print(f"   • Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"   • F1 Score: {f1_weighted:.4f}")
print(f"   • Temps total: {total_train_time:.2f}s ({total_train_time/60:.2f} min)")
print(f"   • Erreurs: {errors.sum()}/{len(y_test)} ({errors.sum()/len(y_test)*100:.2f}%)")

print(f"\n📁 Fichiers générés:")
print(f"   • Modèle: {model_path}")
print(f"   • Résultats: {results_file}")
print(f"   • Visualisations: {results_dir}")
print(f"   • Interprétabilité: {interpretability_dir}")

print("\n🎉 Notebook terminé avec succès!")